# Bioactive Molecule Prediction Using Machine Learning
### Domain: Drug Discovery &middot; Target: Cyclooxygenase-2 (COX-2)

**Objective**:
Given chemical structures represented as SMILES strings, predict whether a molecule is biologically **Active** (1) or **Inactive** (0) against the COX-2 enzyme using supervised machine learning algorithms.

**Pipeline Structure**:
1. Load and inspect preprocessed ChEMBL dataset (`CHEMBL230_Preprocessed_Data.csv`)
2. Feature extraction: Convert SMILES into 2048-bit Morgan circular fingerprints using RDKit
3. Stratified 70% Development / 30% Final Holdout partition
4. 10-Fold Stratified Cross-Validation on the 70% development portion
5. Train all 5 candidate models on full 70% development data
6. Final evaluation on the untouched 30% holdout set
7. Automatic best model selection (Primary: Holdout F1, Secondary: ROC-AUC)
8. Model persistence (`backend/model/best_model.pkl` & `metadata.json`)
9. End-to-end verification and sample prediction

## 2. Import Libraries
Importing core scientific computing, machine learning, chemistry, and visualization packages.

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# RDKit for cheminformatics feature generation
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")  # Suppress chemistry parsing warnings

# Scikit-learn algorithms and evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier

# XGBoost
import xgboost as xgb

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("All libraries imported successfully!")

## 3. Load Preprocessed Dataset
We load the already preprocessed ChEMBL bioactivity dataset. No raw data cleaning or label modification is required.

In [ ]:
DATA_PATH = "CHEMBL230_Preprocessed_Data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded from: {DATA_PATH}")

## 4. Basic Dataset Inspection
Inspecting dimensions, columns, sample rows, and class balance.

In [ ]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
display(df.head())

# Target distribution
class_counts = df["bioactivity_class"].value_counts()
print("\nTarget Class Distribution:")
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(6, 4))
colors = ["#1a6fdb", "#d97706"]
sns.barplot(x=class_counts.index, y=class_counts.values, palette=colors)
plt.title("Bioactivity Class Distribution (COX-2 Dataset)", fontsize=13, fontweight="bold")
plt.xlabel("Bioactivity Class")
plt.ylabel("Number of Molecules")
for i, v in enumerate(class_counts.values):
    plt.text(i, v + 50, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Identify Input and Target (Data Leakage Prevention)
- **Molecular Input**: `smiles` (chemical structure line notation)
- **Target**: `bioactivity_class` (`active` vs `inactive`)
- **Important**: `standard_value` ($IC_{50}$ in nM) is **strictly excluded** from features because `bioactivity_class` was derived from it. Using it would cause severe **data leakage**.

In [ ]:
# Ensure non-null SMILES
initial_count = len(df)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)
print(f"Molecules retained after verifying SMILES: {len(df)} (Dropped: {initial_count - len(df)})")
print("Verification: 'standard_value' is excluded from input features to prevent data leakage.")

## 6. Generate Molecular Features via RDKit
Converting chemical SMILES strings into **Morgan Circular Fingerprints** (radius=2, 2048 bits).
- Morgan fingerprints capture circular atomic environments up to a bond radius of 2 (analogous to ECFP4).
- Produces a 2048-dimensional binary vector for each molecule.

In [ ]:
def smiles_to_fingerprint(smiles: str, radius: int = 2, n_bits: int = 2048):
    """Converts a SMILES string into a 2048-bit Morgan circular fingerprint."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp, dtype=np.uint8)

fingerprints = []
valid_indices = []

for idx, row in df.iterrows():
    fp = smiles_to_fingerprint(row["smiles"])
    if fp is not None:
        fingerprints.append(fp)
        valid_indices.append(idx)

df_valid = df.iloc[valid_indices].reset_index(drop=True)
X = np.array(fingerprints)
print(f"Generated Morgan fingerprints for {len(X)} molecules.")
print(f"Feature matrix X shape: {X.shape} (molecules x 2048 features)")

## 7. Prepare X and y
Encoding target classes:
- `active` $\rightarrow$ **1**
- `inactive` $\rightarrow$ **0**

In [ ]:
label_mapping = {"active": 1, "inactive": 0}
y = df_valid["bioactivity_class"].map(label_mapping).values

print("Label Mapping:", label_mapping)
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Class counts -> Active (1): {np.sum(y == 1)}, Inactive (0): {np.sum(y == 0)}")

## 8. 70/30 Stratified Split
Following the base paper methodology:
- **70% Development Data**: Used for 10-fold cross-validation and full model training.
- **30% Final Holdout Data**: Untouched and reserved strictly for final model evaluation.

In [ ]:
X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Development Set (70%): {X_dev.shape[0]} samples")
print(f"Final Holdout Set (30%): {X_hold.shape[0]} samples (strictly untouched during CV)")

## 9. Define the Five Candidate Models
Architectures evaluated in the base paper:
1. **XGBoost**: Extreme Gradient Boosting (n_estimators=150, max_depth=6, learning_rate=0.15)
2. **Random Forest**: Bagging ensemble of 100 decision trees
3. **Linear SVM**: Linear Support Vector Classifier with L2 penalty
4. **Naive Bayes**: Gaussian Naive Bayes baseline
5. **RBF Network**: Radial Basis Function feature approximation (`RBFSampler`) + logistic classifier

In [ ]:
def get_model_instances():
    """Returns fresh instances of the 5 candidate models."""
    return {
        "XGBoost": xgb.XGBClassifier(
            n_estimators=150,
            max_depth=6,
            learning_rate=0.15,
            eval_metric="logloss",
            random_state=42,
            verbosity=0
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),
        "Linear SVM": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("clf", LinearSVC(C=1.0, max_iter=2000, random_state=42))
        ]),
        "Naive Bayes": GaussianNB(),
        "RBF Network": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("rbf_sampler", RBFSampler(gamma=0.01, n_components=500, random_state=42)),
            ("clf", SGDClassifier(loss="log_loss", max_iter=1000, random_state=42))
        ])
    }

print("5 Model architectures defined.")

## 10. 10-Fold Stratified Cross-Validation on Development Set
Running 10-Fold Stratified Cross-Validation exclusively on the 70% development portion.

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = {}
print("Running 10-Fold Stratified Cross-Validation on 70% Development Data...\n")

for name in get_model_instances().keys():
    fold_f1_scores = []
    
    for train_idx, val_idx in skf.split(X_dev, y_dev):
        X_tr, X_val = X_dev[train_idx], X_dev[val_idx]
        y_tr, y_val = y_dev[train_idx], y_dev[val_idx]
        
        clf = get_model_instances()[name]
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_val)
        fold_f1_scores.append(f1_score(y_val, preds, zero_division=0))
        
    mean_f1 = float(np.mean(fold_f1_scores))
    std_f1 = float(np.std(fold_f1_scores))
    cv_results[name] = {"mean_f1": mean_f1, "std_f1": std_f1}
    print(f"[{name}] Mean 10-Fold CV F1: {mean_f1:.4f} (+/- {std_f1:.4f})")

## 11. Display Mean CV Results
Cross-validation performance comparison table and chart.

In [ ]:
cv_df = pd.DataFrame([
    {"Model": name, "Mean 10-Fold CV F1": data["mean_f1"], "Std Dev": data["std_f1"]}
    for name, data in cv_results.items()
]).sort_values(by="Mean 10-Fold CV F1", ascending=False).reset_index(drop=True)

display(cv_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=cv_df, x="Model", y="Mean 10-Fold CV F1", palette="Blues_r")
plt.title("10-Fold Cross-Validation Performance (Development Set)", fontsize=13, fontweight="bold")
plt.ylim(0.4, 1.0)
for i, row in cv_df.iterrows():
    plt.text(i, row["Mean 10-Fold CV F1"] + 0.015, f"{row['Mean 10-Fold CV F1']:.4f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 12. Train Each Model on Complete 70% Development Data
Training each final candidate model on all 4,787 development samples.

In [ ]:
trained_models = {}

print("Fitting each candidate model on full 70% development set...")
for name, model in get_model_instances().items():
    model.fit(X_dev, y_dev)
    trained_models[name] = model
    print(f"  [OK] Trained {name}")

## 13. Evaluate on Untouched 30% Final Holdout Set
Evaluating models on the 2,052 unseen holdout molecules.
- **Sensitivity (Recall)**: $\frac{TP}{TP + FN}$
- **Specificity**: $\frac{TN}{TN + FP}$
- **Paper AUC (Balanced Accuracy)**: $\frac{\text{Sensitivity} + \text{Specificity}}{2}$
- **ROC-AUC (Modern)**: Computed from continuous prediction probabilities

In [ ]:
holdout_metrics = []
confusion_matrices = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_hold)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_hold)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_hold)
        y_proba = 1.0 / (1.0 + np.exp(-scores))
    else:
        y_proba = None
        
    acc = accuracy_score(y_hold, y_pred)
    prec = precision_score(y_hold, y_pred, zero_division=0)
    rec = recall_score(y_hold, y_pred, zero_division=0)
    f1 = f1_score(y_hold, y_pred, zero_division=0)
    
    cm = confusion_matrix(y_hold, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    paper_auc = (rec + spec) / 2.0
    roc_auc = roc_auc_score(y_hold, y_proba) if y_proba is not None else 0.0
    
    confusion_matrices[name] = {"cm": cm, "tp": tp, "tn": tn, "fp": fp, "fn": fn}
    
    holdout_metrics.append({
        "Model": name,
        "Mean CV F1": round(cv_results[name]["mean_f1"], 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Sensitivity (Recall)": round(rec, 4),
        "Specificity": round(spec, 4),
        "Holdout F1": round(f1, 4),
        "Paper AUC (Bal Acc)": round(paper_auc, 4),
        "ROC-AUC": round(roc_auc, 4)
    })

print("Holdout evaluation completed for all 5 models.")

## 14. Confusion Matrices (Holdout Evaluation)
Displaying classification counts: True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, (name, data) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    sns.heatmap(data["cm"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Inactive", "Active"], yticklabels=["Inactive", "Active"])
    ax.set_title(f"{name}\nTP={data['tp']}, TN={data['tn']}\nFP={data['fp']}, FN={data['fn']}", fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 15. Final Model Comparison Table (30% Holdout)
Comparing performance metrics across the final holdout set.

In [ ]:
results_table = pd.DataFrame(holdout_metrics).sort_values(by=["Holdout F1", "ROC-AUC"], ascending=[False, False]).reset_index(drop=True)
display(results_table)

## 16. Automatic Best Model Selection
Selection Criteria:
1. **Primary**: Highest Holdout F1-Score
2. **Tie-breaker**: Highest ROC-AUC

In [ ]:
best_row = results_table.iloc[0]
best_model_name = best_row["Model"]
best_model_instance = trained_models[best_model_name]

print("=" * 55)
print(f"AUTOMATICALLY SELECTED BEST MODEL: {best_model_name}")
print("=" * 55)
print(f"Selection Rule : Highest Holdout F1-Score (Tie-breaker: ROC-AUC)")
print(f"Holdout F1     : {best_row['Holdout F1']:.4f}")
print(f"ROC-AUC        : {best_row['ROC-AUC']:.4f}")
print(f"Accuracy       : {best_row['Accuracy']*100:.2f}%")
print(f"Mean 10-Fold CV: {best_row['Mean CV F1']:.4f}")
print(f"Paper AUC      : {best_row['Paper AUC (Bal Acc)']:.4f}")
print("=" * 55)

## 17. Save Best Model and Metadata
Saving `best_model.pkl` and `metadata.json` to `backend/model/` for use by the FastAPI service.

In [ ]:
MODEL_DIR = os.path.join("backend", "model")
os.makedirs(MODEL_DIR, exist_ok=True)

model_save_path = os.path.join(MODEL_DIR, "best_model.pkl")
joblib.dump(best_model_instance, model_save_path)
print(f"Saved winning model to: {model_save_path}")

metadata = {
    "best_model_name": best_model_name,
    "label_mapping": label_mapping,
    "reverse_mapping": {"1": "active", "0": "inactive"},
    "fingerprint_radius": 2,
    "fingerprint_bits": 2048,
    "split": "70% development / 30% holdout",
    "cv_folds": 10,
    "selection_metric": "Holdout F1-Score",
    "best_metrics": {
        "cv_f1_mean": float(best_row["Mean CV F1"]),
        "holdout_f1": float(best_row["Holdout F1"]),
        "accuracy": float(best_row["Accuracy"]),
        "roc_auc": float(best_row["ROC-AUC"]),
        "paper_auc": float(best_row["Paper AUC (Bal Acc)"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Sensitivity (Recall)"]),
        "specificity": float(best_row["Specificity"])
    },
    "all_results": results_table.to_dict(orient="records")
}

meta_save_path = os.path.join(MODEL_DIR, "metadata.json")
with open(meta_save_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to: {meta_save_path}")

## 18. Test Example Predictions
Testing live inference using the saved best model on known active and inactive molecules.

In [ ]:
loaded_model = joblib.load(model_save_path)

test_molecules = {
    "Aspirin (weak/inactive COX-2)": "CC(=O)Oc1ccccc1C(=O)O",
    "Potent COX-2 Inhibitor (Active)": "CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1",
    "Ethanol (Inactive)": "CCO"
}

print("Running test predictions through loaded model:")
print("-" * 65)

for name, smiles in test_molecules.items():
    fp = smiles_to_fingerprint(smiles)
    if fp is not None:
        fp_reshaped = fp.reshape(1, -1)
        pred_class = int(loaded_model.predict(fp_reshaped)[0])
        label = "Active" if pred_class == 1 else "Inactive"
        
        if hasattr(loaded_model, "predict_proba"):
            prob = float(loaded_model.predict_proba(fp_reshaped)[0][pred_class])
        else:
            prob = 1.0
            
        print(f"Molecule   : {name}")
        print(f"SMILES     : {smiles}")
        print(f"Prediction : {label} (Confidence: {prob*100:.2f}%)")
        print("-" * 65)

## 19. Final Results Summary
- **Evaluation Protocol**: 70% Development (10-fold CV) + 30% Holdout evaluation.
- **Top Performer**: The automatically selected best model achieved top F1 and ROC-AUC metrics.
- **FastAPI Deployment**: Persisted model artifact is served live via FastAPI at `http://127.0.0.1:8000/app/`.

In [ ]:
print("ML Pipeline Notebook execution complete!")
print(f"Active Model deployed: {best_model_name}")